# Boss's Orders: Human-in-the-Loop Document & Email Drafting Agent

### Problem Statement:
> *"Our company is not working efficiently! We spend way too much time drafting documents and this needs to be fixed! For the company, you need to create an AI Agentic System that can speed up drafting documents, emails, etc. The AI Agentic System should have Human-AI Collaboration meaning the Human should be able to provide continuous feedback and the AI Agent should stop when the Human is happy with the draft. The system should also be fast and be able to save the drafts."*

### System Architecture & Key Capabilities:
1. **Fast AI Drafter (`drafting_node`)**: Rapidly generates and refines documents (proposals, executive updates, client emails, memos) using GPT-4o based on intent and iterative feedback.
2. **Human-in-the-Loop Review (`human_review_node`)**: The human reviews the draft and provides approval or specific modification instructions.
3. **Feedback Routing Loop (`check_human_feedback`)**: Loops back to the drafting node if the human requests changes; routes to draft saving if the human approves (`"approve"`, `"looks good"`, `"yes"`, `"done"`).
4. **Persistent Draft Storage (`save_draft_node`)**: Automatically saves the final approved draft as a markdown/text file with metadata (topic, version count, timestamp) to the `saved_drafts/` directory.
5. **State Management**: Uses LangGraph's `StateGraph` and checkpoint memory (`MemorySaver`) to remember the full iteration history.

## 0. Dependencies and Setup

In [ ]:
# Install dependencies if needed:
# !pip install langgraph langchain-core langchain-openai python-dotenv

In [ ]:
import os
import re
from datetime import datetime
from typing import Annotated, List, TypedDict, Optional, Literal
from dotenv import load_dotenv

# LangChain Message & Model abstractions
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    AIMessage,
    SystemMessage
)
from langchain_openai import ChatOpenAI

# LangGraph State & Checkpointer
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

## 1. Define State Structure

The `DraftState` tracks:
- `document_type`: Type of document (e.g., `"Email"`, `"Proposal"`, `"Memo"`, `"Announcement"`).
- `topic`: Subject of the document.
- `audience`: Target readers (e.g., `"Engineering Team"`, `"C-Suite"`, `"Clients"`).
- `key_points`: Bullet points or core context to include.
- `current_draft`: The latest draft content.
- `draft_history`: List of prior drafts for review and comparison.
- `human_feedback`: Feedback provided by the human reviewer on the current draft.
- `is_approved`: Boolean flag indicating human approval.
- `revision_count`: Counter of revisions made.
- `saved_file_path`: Path where the approved draft is saved.

In [ ]:
class DraftState(TypedDict, total=False):
    document_type: str
    topic: str
    audience: str
    key_points: List[str]
    current_draft: str
    draft_history: List[str]
    human_feedback: str
    is_approved: bool
    revision_count: int
    saved_file_path: str

## 2. Initialize Fast AI Drafter (GPT-4o)

We use `gpt-4o` with a low temperature for rapid, structured corporate document drafting.

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.3
)
print("Drafter LLM Initialized:", llm.model_name)

## 3. Node Definitions: Drafting, Review & Saving

In [ ]:
DRAFTER_SYSTEM_PROMPT = """You are an executive communication and document drafting expert for a fast-paced company.
Your objective is to produce professional, clean, concise, and persuasive documents and emails.
When incorporating human feedback, maintain the core facts while directly addressing the reviewer's critiques and refinements."""

def drafter_node(state: DraftState) -> dict:
    """
    Generates an initial draft or revises the current draft according to human feedback.
    """
    doc_type = state.get("document_type", "Document")
    topic = state.get("topic", "Corporate Communication")
    audience = state.get("audience", "Team Members")
    key_points = state.get("key_points", [])
    current_draft = state.get("current_draft", "")
    feedback = state.get("human_feedback", "")
    revision = state.get("revision_count", 0) + 1
    draft_history = list(state.get("draft_history", []))
    
    if current_draft:
        draft_history.append(current_draft)
        prompt = f"""
We have an existing draft that requires revision based on Human Feedback.

Document Type: {doc_type}
Topic: {topic}
Audience: {audience}

--- CURRENT DRAFT ---
{current_draft}

--- HUMAN FEEDBACK ---
{feedback}

Please rewrite and improve the draft, incorporating all the requested feedback precisely.
Return only the revised draft content.
"""
    else:
        formatted_points = "\n".join([f"- {pt}" for pt in key_points]) if key_points else "Standard business format."
        prompt = f"""
Draft a professional {doc_type}.
Topic: {topic}
Target Audience: {audience}
Key Points to Include:
{formatted_points}

Ensure clarity, executive tone, and actionable structure.
Return only the draft content.
"""

    response = llm.invoke([
        SystemMessage(content=DRAFTER_SYSTEM_PROMPT),
        HumanMessage(content=prompt)
    ])
    
    new_draft = response.content.strip()
    
    return {
        "current_draft": new_draft,
        "draft_history": draft_history,
        "revision_count": revision,
        "human_feedback": ""  # Reset feedback after drafting
    }

def save_draft_node(state: DraftState) -> dict:
    """
    Saves the approved draft to the file system.
    """
    os.makedirs("saved_drafts", exist_ok=True)
    
    topic = state.get("topic", "draft").strip().lower()
    safe_topic = re.sub(r'[^a-z0-9]+', '_', topic)[:30]
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"saved_drafts/{safe_topic}_{timestamp}.md"
    
    content = f"# Approved Document: {state.get('topic')}\n"
    content += f"- **Document Type**: {state.get('document_type')}\n"
    content += f"- **Audience**: {state.get('audience')}\n"
    content += f"- **Revisions Made**: {state.get('revision_count', 1)}\n"
    content += f"- **Saved At**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += "## Content\n\n"
    content += state.get("current_draft", "")
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
        
    print(f"\n>>> Success: Approved draft saved to '{filename}'")
    return {"saved_file_path": filename, "is_approved": True}

## 4. Conditional Edge: Human Approval Check

The routing logic checks `is_approved`:
- If the human is happy $\rightarrow$ proceed to `save_draft` $\rightarrow$ `END`.
- If changes are requested $\rightarrow$ loop back to `drafter`.

In [ ]:
def route_after_human_feedback(state: DraftState) -> Literal["save_draft", "drafter"]:
    if state.get("is_approved", False):
        return "save_draft"
    return "drafter"

## 5. Build and Compile the Drafting Graph

In [ ]:
builder = StateGraph(DraftState)

# Add nodes
builder.add_node("drafter", drafter_node)
builder.add_node("save_draft", save_draft_node)

# Start with drafting
builder.add_edge(START, "drafter")

# Conditional routing after drafting
builder.add_conditional_edges(
    "drafter",
    route_after_human_feedback,
    {
        "save_draft": "save_draft",
        "drafter": "drafter"
    }
)

builder.add_edge("save_draft", END)

# Use MemorySaver checkpointer for Human-in-the-loop state persistence across turns
checkpointer = MemorySaver()
drafting_agent = builder.compile(checkpointer=checkpointer)
print("Drafting Agent Graph compiled with persistent memory!")

## 6. Interactive Human-in-the-Loop Collaboration Function

This function drives the Human-AI collaborative process:
1. AI produces the initial draft quickly.
2. Human reviews the draft:
   - Typing `"approve"`, `"looks good"`, `"yes"`, or `"ok"` signals approval.
   - Typing specific suggestions (e.g. *"make it more polite"*, *"add a deadline of Friday 5pm"*) triggers AI revision.
3. The loop stops as soon as the human is satisfied, saving the draft automatically.

In [ ]:
def start_collaborative_drafting_session(
    document_type: str,
    topic: str,
    audience: str,
    key_points: List[str],
    thread_id: str = "draft_session_001"
):
    """
    Runs a live Human-in-the-Loop drafting session.
    """
    config = {"configurable": {"thread_id": thread_id}}
    
    initial_input = {
        "document_type": document_type,
        "topic": topic,
        "audience": audience,
        "key_points": key_points,
        "human_feedback": "",
        "is_approved": False,
        "revision_count": 0
    }
    
    print(f"\n{'='*75}")
    print(f"  AI AGENTIC DRAFTING SYSTEM - HUMAN-AI COLLABORATION SESSION")
    print(f"  Topic: {topic} | Type: {document_type} | Audience: {audience}")
    print(f"{'='*75}\n")
    
    try:
        # First turn: AI generates draft
        state = drafting_agent.invoke(initial_input, config=config)
    except Exception as e:
        print(f"Execution Note (API key required for live execution): {e}")
        return
        
    while not state.get("is_approved", False):
        current_draft = state.get("current_draft", "")
        revision = state.get("revision_count", 1)
        
        print(f"\n----------------- CURRENT DRAFT (Revision #{revision}) -----------------")
        print(current_draft)
        print("------------------------------------------------------------------\n")
        
        # Human Review Feedback Prompt
        print("Options: Type 'approve' (or 'looks good') to approve and save.")
        print("         OR enter your specific feedback/edits to refine the draft.")
        
        user_input = input("Human Feedback: ").strip()
        if not user_input:
            continue
            
        # Check for approval
        if user_input.lower() in ["approve", "looks good", "yes", "ok", "done", "perfect"]:
            print("\nHuman approved the draft! Proceeding to save...")
            state = drafting_agent.invoke({
                "is_approved": True
            }, config=config)
            break
        else:
            print(f"\nSending feedback to AI for revision #{revision + 1}...")
            state = drafting_agent.invoke({
                "human_feedback": user_input,
                "is_approved": False
            }, config=config)
            
    print("\nSession completed successfully!")
    if state.get("saved_file_path"):
        print(f"Final file location: {state.get('saved_file_path')}")

## 7. Automated Simulation of Human-AI Collaboration

To demonstrate the loop without requiring real-time terminal input, here is an automated end-to-end simulation:
- **Turn 1**: Initial Draft generation for a Project Deadline extension email.
- **Turn 2**: Human provides feedback: *"Please make the tone more apologetic, highlight that testing will be thorough, and mention the new deadline is Friday at 5 PM EST."*
- **Turn 3**: Human approves: *"Looks good! Approve."*

In [ ]:
sim_config = {"configurable": {"thread_id": "simulation_demo_01"}}

print("=== SIMULATION TURN 1: Initial AI Draft Generation ===")
turn1_input = {
    "document_type": "Client Email",
    "topic": "Project Alpha Release Date Adjustment",
    "audience": "Executive Stakeholders at Acme Corp",
    "key_points": [
        "We discovered an edge-case performance bug in the payment gateway.",
        "Need 48 hours for rigorous load testing.",
        "New target deployment: Friday 5 PM EST."
    ],
    "human_feedback": "",
    "is_approved": False,
    "revision_count": 0
}

try:
    s1 = drafting_agent.invoke(turn1_input, config=sim_config)
    print("\n--- Initial Draft ---\n", s1["current_draft"])
    
    print("\n=== SIMULATION TURN 2: Human Feedback Provided ===")
    feedback = "Make the opening tone more empathetic and reassure them that security is our #1 priority."
    print(f"Human Feedback: '{feedback}'")
    
    s2 = drafting_agent.invoke({"human_feedback": feedback, "is_approved": False}, config=sim_config)
    print("\n--- Revised Draft (Revision #2) ---\n", s2["current_draft"])
    
    print("\n=== SIMULATION TURN 3: Human Approves Draft ===")
    print("Human Feedback: 'Looks great! Approve and save.'")
    s3 = drafting_agent.invoke({"is_approved": True}, config=sim_config)
    print(f"Final Status: Approved = {s3.get('is_approved')}")
    print(f"Saved File Path = {s3.get('saved_file_path')}")
except Exception as e:
    print(f"Simulation Note (Live API key required): {e}")